# Text Preprocessing

**Course:** [Natural Language Processing](https://ml-viz-ruby.vercel.app/courses/nlp/01-text-preprocessing)

This notebook implements tokenization strategies (BPE, word-level, character-level), stemming vs lemmatization, and TF-IDF from scratch.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import re
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## Tokenization strategies

Three granularities: character, word, and subword. The key trade-off is vocabulary size vs sequence length.

In [ ]:
sentence = "The quick brown fox jumps over the lazy dog"

# Character-level
char_tokens = list(sentence)

# Word-level
word_tokens = sentence.lower().split()

print(f"Character-level: {len(char_tokens)} tokens")
print(f"Word-level:      {len(word_tokens)} tokens")
print(f"\nChar tokens[:10]: {char_tokens[:10]}")
print(f"Word tokens:      {word_tokens}")

## Byte-Pair Encoding (BPE) — from scratch

BPE iteratively merges the most frequent adjacent pair of symbols, building a subword vocabulary from character-level up.

In [ ]:
def get_vocab(corpus):
    """Build initial vocab: each word split into chars + end-of-word marker."""
    vocab = defaultdict(int)
    for word in corpus.lower().split():
        vocab[' '.join(list(word)) + ' </w>'] += 1
    return dict(vocab)

def get_pairs(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return dict(pairs)

def merge_vocab(pair, vocab):
    merged = defaultdict(int)
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        merged[new_word] += vocab[word]
    return dict(merged)

corpus = "low lower newest widest lowest lowest newest"
vocab = get_vocab(corpus)
print("Initial vocabulary:")
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f"  '{word}': {freq}")

In [ ]:
# Run 8 BPE merges
merge_ops = []
for step in range(8):
    pairs = get_pairs(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    merge_ops.append((best, pairs[best]))
    vocab = merge_vocab(best, vocab)
    print(f"Step {step+1}: merge {best} (freq={pairs[best]})")

print("\nFinal vocabulary:")
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f"  '{word}': {freq}")

## Visualizing merge frequency over steps

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
labels = [f"('{m[0]}','{m[1]}')".replace("</w>", "⏎") for (m, _) in merge_ops]
freqs = [f for (_, f) in merge_ops]
bars = ax.bar(range(len(freqs)), freqs, color='#6366f1', alpha=0.85)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Merge frequency')
ax.set_title('BPE merge operations (frequency at time of merge)', fontsize=12)
plt.tight_layout()
plt.show()

## TF-IDF from scratch

TF-IDF = Term Frequency × Inverse Document Frequency. Terms ubiquitous across all documents score 0.

In [ ]:
documents = [
    "machine learning is a subfield of artificial intelligence",
    "deep learning uses neural networks for machine learning tasks",
    "natural language processing is a machine learning application",
    "neural networks learn representations through gradient descent",
    "transformers revolutionized natural language processing",
]

def tokenize(doc):
    return doc.lower().split()

def compute_tf(doc_tokens):
    counts = Counter(doc_tokens)
    total = len(doc_tokens)
    return {w: c / total for w, c in counts.items()}

def compute_idf(docs_tokens):
    N = len(docs_tokens)
    df = defaultdict(int)
    for tokens in docs_tokens:
        for t in set(tokens):
            df[t] += 1
    return {t: math.log(N / (1 + df[t])) for t in df}

tokenized = [tokenize(d) for d in documents]
idf = compute_idf(tokenized)

# Show top TF-IDF terms per document
for i, (doc, tokens) in enumerate(zip(documents[:3], tokenized[:3])):
    tf = compute_tf(tokens)
    tfidf = {w: tf[w] * idf[w] for w in tf}
    top = sorted(tfidf.items(), key=lambda x: -x[1])[:3]
    print(f"Doc {i+1}: '{doc[:40]}...'")
    print(f"  Top terms: {top}\n")

In [ ]:
# Visualize TF-IDF heatmap for a subset of terms
interesting_words = ["machine", "learning", "neural", "transformers", "language", "networks"]
matrix = []
for tokens in tokenized:
    tf = compute_tf(tokens)
    row = [tf.get(w, 0) * idf.get(w, 0) for w in interesting_words]
    matrix.append(row)

matrix = np.array(matrix)
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(matrix, cmap='magma', aspect='auto')
ax.set_xticks(range(len(interesting_words)))
ax.set_xticklabels(interesting_words, rotation=30, ha='right')
ax.set_yticks(range(len(documents)))
ax.set_yticklabels([f"Doc {i+1}" for i in range(len(documents))])
ax.set_title('TF-IDF scores', fontsize=12)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## ✏️ Your turn

### Exercise 1: Implement BPE tokenization for a new word

Given a learned set of BPE merge rules, tokenize an unseen word.

The merge rules are applied left-to-right in order of when they were learned (highest-frequency merges first).

In [ ]:
def bpe_tokenize(word, merge_rules):
    """
    Tokenize a word using learned BPE merge rules.
    
    Args:
        word: str, input word (e.g., 'newer')
        merge_rules: list of (pair_tuple) in priority order
                     e.g. [('e', 's'), ('n', 'e'), ('ne', 'w')]
    Returns:
        list of subword strings
    """
    # Start with character-level split + end-of-word marker
    tokens = list(word) + ['</w>']
    
    # TODO(you): for each merge rule (a, b), find any adjacent (a, b) pair
    # in tokens and replace it with the merged 'ab' token.
    # Apply rules in order; restart from the beginning after each merge.
    
    return tokens


# Test rules learned from 'low lower newest widest'
learned_rules = [('e', 's'), ('es', 't'), ('est', '</w>'), ('l', 'o'), ('lo', 'w'),
                 ('n', 'e'), ('ne', 'w'), ('new', 'est</w>')]

result = bpe_tokenize('newest', learned_rules)
print('Tokenization of "newest":', result)

In [ ]:
# Assertion: 'newest' should merge into 'newest</w>' in one or two pieces
result = bpe_tokenize('newest', learned_rules)
assert 'newest</w>' in result or result == ['n', 'e', 'w', 'e', 's', 't', '</w>'] or len(result) < 8, \
    f"Expected some merges to occur, got {result}"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bpe_tokenize(word, merge_rules):
    tokens = list(word) + ['</w>']
    for (a, b) in merge_rules:
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens
```
</details>

### Exercise 2: TF-IDF for a new document

Compute TF-IDF scores for all unique terms in a new document, using IDF values computed from the existing corpus above.

In [ ]:
def tfidf_for_doc(doc_string, precomputed_idf):
    """
    Compute TF-IDF scores for all tokens in doc_string.
    
    Args:
        doc_string: str, the new document
        precomputed_idf: dict mapping term -> IDF score
    Returns:
        dict mapping term -> TF-IDF score (skip terms with IDF=0)
    """
    # TODO(you): tokenize the document, compute TF, multiply by IDF
    # Skip any term not in precomputed_idf (unseen in training corpus)
    pass


new_doc = "language models learn machine representations"
scores = tfidf_for_doc(new_doc, idf)
if scores:
    print("TF-IDF scores:", sorted(scores.items(), key=lambda x: -x[1]))

In [ ]:
scores = tfidf_for_doc(new_doc, idf)
assert isinstance(scores, dict), "Should return a dict"
assert all(isinstance(v, float) for v in scores.values()), "Values should be floats"
# 'machine' appears in 3/5 docs → lower IDF; 'representations' in 1/5 → higher IDF
if 'representations' in scores and 'machine' in scores:
    assert scores['representations'] > scores['machine'], \
        "'representations' (rare) should score higher than 'machine' (common)"

# Edge case: an empty document has no tokens, so it should score as an empty dict
# (not raise a ZeroDivisionError from compute_tf).
empty_scores = tfidf_for_doc("", idf)
assert empty_scores == {}, "An empty document should yield an empty TF-IDF dict"

# Edge case: a single-word document made only of out-of-vocabulary terms
oov_scores = tfidf_for_doc("zzzzzz", idf)
assert oov_scores == {}, "A document of only unseen terms should yield an empty dict"

# Edge case: a single-word document whose one word IS in the training corpus —
# TF is 1.0 for that word, so TF-IDF collapses to exactly its IDF.
single_scores = tfidf_for_doc("machine", idf)
assert single_scores == {"machine": idf["machine"]}, \
    "TF-IDF of a single-word doc should equal that word's IDF"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def tfidf_for_doc(doc_string, precomputed_idf):
    tokens = doc_string.lower().split()
    tf = compute_tf(tokens)
    return {t: tf[t] * precomputed_idf[t] for t in tf if t in precomputed_idf}
```
</details>

---
### Extra practice — DML #129: Unigram probability from a corpus

This is the [Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem) formulation of the counting problem underneath every n-gram language model: given a corpus of sentences wrapped in `<s>` / `</s>` boundary tokens, compute
$P(\text{word}) = \text{count(word)} / \text{total token count}$ — where the boundary tokens themselves count as tokens. Round to 4 decimal places.

In [ ]:
def unigram_probability(corpus: str, word: str) -> float:
    """
    DML #129 -- unigram probability of `word` in a whitespace-tokenized `corpus`
    that includes <s>/</s> sentence-boundary tokens.

    Args:
        corpus: str, e.g. "<s> Jack I like </s> <s> Jack I do like </s>"
        word: str, the token to compute P(word) for
    Returns:
        float, rounded to 4 decimal places
    """
    tokens = corpus.split()

    # TODO(you): total number of tokens (including <s>/</s>)
    total = ...

    # TODO(you): how many times `word` occurs among `tokens`
    count = ...

    return round(count / total, 4)


corpus_129 = "<s> Jack I like </s> <s> Jack I do like </s>"
print(unigram_probability(corpus_129, "Jack"))

In [ ]:
# Checks — run me (DML's own test cases)
assert unigram_probability(corpus_129, "Jack") == 0.1818

corpus_a = "<s> I am Jack </s> <s> Jack I am </s> <s> Jack I like </s> <s> Jack I do like </s> <s> do I like Jack </s>"
assert unigram_probability(corpus_a, "Jack") == 0.1852
assert unigram_probability(corpus_a, "like") == 0.1111

corpus_b = "<s> hello world </s> <s> hello </s>"
assert unigram_probability(corpus_b, "hello") == 0.2857

# Edge case: single-word corpus (one sentence, one content word between the markers)
corpus_single = "<s> hello </s>"
assert unigram_probability(corpus_single, "hello") == 0.3333  # 1 / 3 tokens

# Edge case: a word that never appears in the corpus
assert unigram_probability(corpus_single, "missing") == 0.0

print("✅ DML #129 passed")

<details>
<summary>💡 Show solution</summary>

```python
def unigram_probability(corpus: str, word: str) -> float:
    tokens = corpus.split()
    total = len(tokens)
    count = tokens.count(word)
    return round(count / total, 4)
```
</details>

### Extra practice — DML #51: Optimal String Alignment (edit) distance

This is the [Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem) formulation of edit distance with a spell-check-relevant twist: on top of insert / delete / substitute, an **adjacent transposition** (swapping two neighboring characters, e.g. `teh` → `the`) also costs only 1 — the [Damerau–Levenshtein](https://en.wikipedia.org/wiki/Damerau%E2%80%93Levenshtein_distance) variant. That is exactly the distance metric a spell-checker built on top of the tokenizer above would want, since transposed letters are among the most common typos.

In [ ]:
def OSA(source: str, target: str) -> int:
    """
    DML #51 -- Optimal String Alignment distance: the minimum number of
    insertions, deletions, substitutions, and adjacent transpositions
    (each costing 1) needed to turn `source` into `target`.
    """
    n, m = len(source), len(target)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(n + 1):
        dp[i][0] = i

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # TODO(you): substitution cost is 0 if the chars match, else 1
            sub_cost = ...
            dp[i][j] = min(
                dp[i - 1][j] + 1,              # deletion
                dp[i][j - 1] + 1,              # insertion
                dp[i - 1][j - 1] + sub_cost,   # substitution / match
            )
            # TODO(you): if the last two chars of source/target are a swapped
            # pair, also allow a transposition from dp[i-2][j-2]
            if i > 1 and j > 1 and source[i - 1] == target[j - 2] and source[i - 2] == target[j - 1]:
                dp[i][j] = min(dp[i][j], ...)

    return dp[n][m]


print(OSA("caper", "acer"))

In [ ]:
# Checks — run me (DML's own test cases)
assert OSA("butterfly", "dragonfly") == 6
assert OSA("caper", "acer") == 2
assert OSA("telescope", "microscope") == 5
assert OSA("london", "paris") == 6

# Edge cases
assert OSA("", "") == 0                 # two empty strings need no edits
assert OSA("", "abc") == 3              # empty source -> 3 insertions
assert OSA("abc", "") == 3              # empty target -> 3 deletions
assert OSA("teh", "the") == 1           # a single adjacent transposition
assert OSA("kitten", "kitten") == 0     # identical strings -> distance 0

print("✅ DML #51 passed")

<details>
<summary>💡 Show solution</summary>

```python
def OSA(source: str, target: str) -> int:
    n, m = len(source), len(target)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(n + 1):
        dp[i][0] = i
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub_cost = 0 if source[i - 1] == target[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + sub_cost,
            )
            if i > 1 and j > 1 and source[i - 1] == target[j - 2] and source[i - 2] == target[j - 1]:
                dp[i][j] = min(dp[i][j], dp[i - 2][j - 2] + 1)
    return dp[n][m]
```
</details>